In [65]:
import os
import nbformat
import re
import pandas as pd

# Define the directory containing your notebooks
directory = 'Results'

# Initialize an empty list to hold your results
results = []

# Iterate over all files in the directory
for filename in os.listdir(directory):
    # Check if this is a .ipynb file
    if filename.endswith('.ipynb'):
        # Read the notebook
        with open(os.path.join(directory, filename)) as f:
            nb = nbformat.read(f, as_version=4)
            
            # Initialize variables
            model_name = ''
            cpu_time = ''
            wall_time = ''
            
            # Iterate over all cells in the notebook
            for cell in nb.cells:
                # Check if this is an output cell
                if cell.cell_type == 'markdown':
                    model = re.search('^(\d+\..*)', cell.source, re.MULTILINE)
                    if model:
                        model_name = model.group(1)
                if cell.cell_type == 'code':
                    for output in cell.outputs:
                        if 'text' in output:
                            # Look for lines that match the pattern
                            mse = re.search('MSE = ([\d\.]+)', output['text'])
                            rmse = re.search('RMSE = ([\d\.]+)', output['text'])
                            mae = re.search('MAE = ([\d\.]+)', output['text'])
                            cpu = re.search('total: ([\d\.]+ ms)', output['text'])
                            wall = re.search('Wall time: ([\d\w\s\.]+)', output['text'])
                            if mse and rmse and mae and cpu and wall:
                                # If a match is found, store the results
                                results.append({
                                    'notebook': filename[:-6],
                                    'Model Name': model_name,
                                    'MSE': float(mse.group(1)),
                                    'RMSE': float(rmse.group(1)),
                                    'MAE': float(mae.group(1)),
                                    'CPU Time': cpu.group(1),
                                    'Wall Time': wall.group(1)[:-1]
                                })

# Convert the results to a pandas DataFrame
df = pd.DataFrame(results)
# Define the model names
model_names = ['Baseline', 'ALS', 'Similarity']
model_names_magazine = ['Baseline', 'ALS', 'Similarity', 'Sentiment']

# Apply the model names to the 'Model Name' column
df['Model Name'] = df.apply(
    lambda row: model_names_magazine[row.name % len(model_names_magazine)] 
    if row['notebook'] == 'Magazine_Subscriptions (all models)' 
    else model_names[row.name % len(model_names)], 
    axis=1
)

# Convert CPU Time to milliseconds
df['CPU Time'] = df['CPU Time'].apply(
    lambda x: float(x.split()[0]) if 'ms' in x else float(x.split()[0]) * 1000)

# Function to convert time string to seconds
def convert_to_seconds(time_str):
    time_parts = re.findall(r'(\d+\.?\d*)\s*(\w+)', time_str)
    total_seconds = 0
    for time, unit in time_parts:
        if unit == 's':
            total_seconds += float(time)
        elif unit == 'min':
            total_seconds += float(time) * 60
    return total_seconds

df['Wall Time'] = df['Wall Time'].apply(convert_to_seconds)
df

,notebook,Model Name,MSE,RMSE,MAE,CPU Time,Wall Time
0,Software,Baseline,2.401111,1.549552,1.256019,80.7,27.70
1,Software,ALS,5.269411,2.295520,1.940394,387.0,115.00
2,Software,Similarity,2.795662,1.672023,1.262217,197.0,303.00
3,Digital_Music,Baseline,0.737282,0.858651,0.494056,86.1,68.00
4,Digital_Music,ALS,2.546375,1.595737,1.237013,512.0,266.00
5,Digital_Music,Similarity,0.799045,0.893893,0.335644,316.0,666.00
6,Toys_and_Games,Baseline,1.582209,1.257859,0.890716,146.0,298.00
7,All_Beauty,ALS,1.717823,1.310657,0.974093,64.0,19.70
8,All_Beauty,Similarity,5.847922,2.418248,2.097802,258.0,67.00
9,All_Beauty,Baseline,1.777690,1.333300,1.011797,158.0,169.00


In [66]:
# List to store the results
results = []

# Iterate over all files in the directory
for filename in os.listdir(directory):
    # Check if this is a .ipynb file
    if filename.endswith('.ipynb'):
        # Read the notebook
        with open(os.path.join(directory, filename)) as f:
            nb = nbformat.read(f, as_version=4)
            
            # Initialize dataset_size
            dataset_size = ''
            
            # Iterate over all cells in the notebook
            for cell in nb.cells:
                # Check if this is a code cell
                if cell.cell_type == 'code':
                    for output in cell.outputs:
                        if 'text' in output:
                            # Look for lines that match the pattern
                            size = re.search('\(([\d\.]+(?:M|G))', output['text'])
                            if size:
                                dataset_size = size.group(1)
                                # If a match is found, store the results
                                results.append({
                                    'notebook': filename[:-6],
                                    'Dataset Size': dataset_size
                                })

# Convert the results to a pandas DataFrame
df2 = pd.DataFrame(results)
df2

,notebook,Dataset Size
0,Software,94M
1,Digital_Music,207M
2,Toys_and_Games,953M
3,All_Beauty,45M
4,Arts_Crafts_and_Sewing,292M
5,Office_Products,666M
6,Automotive,866M
7,Kindle_Store,1.1G
8,Luxury_Beauty,70M
9,Prime_Pantry,43M
